In [ ]:
import cv2
import pandas as pd
import numpy as np
import albumentations as A
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from tqdm import tqdm
import random

# Config
SEEDS = [42, 123, 593, 2324, 3]
DATA_DIR = Path("starches_dataset")
BASE_OUTPUT_DIR = Path("dataset_splits_5_folds")

LABEL_MAP = {
    "kukurydziana": "corn",
    "pszenna": "wheat",
    "ryzowa": "rice",
    "z_tapioki": "tapioca",
    "ziemniaczana": "potato"
}

transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.06, scale_limit=0.1, rotate_limit=45, border_mode=cv2.BORDER_REFLECT_101, p=0.7),
    A.RandomBrightnessContrast(p=0.5),
    A.GaussianBlur(blur_limit=3, p=0.2),
])

def process_and_save(df_subset, split_name, out_base_dir, augment=False, aug_multiplier=3):
    out_split_dir = out_base_dir / split_name
    
    for idx, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc=f"Saving {split_name} set", leave=False):
        img_path = str(row['filepath'])
        label = row['label']
        group_id = str(row['group_id'])
        
        out_label_dir = out_split_dir / label
        out_label_dir.mkdir(parents=True, exist_ok=True)
        
        try:
            img_array = np.fromfile(img_path, np.uint8)
            img = cv2.imdecode(img_array, cv2.IMREAD_GRAYSCALE)
        except Exception:
            continue
            
        if img is None:
            continue
            
        base_name = Path(img_path).stem
        safe_group_id = "".join([c if c.isalnum() else "_" for c in group_id])
        unique_base_name = f"{safe_group_id}_{base_name}"
        
        orig_out_path = str(out_label_dir / f"{unique_base_name}_orig.jpg")
        success, buffer = cv2.imencode('.jpg', img)
        if success: buffer.tofile(orig_out_path)
        
        if augment:
            for i in range(aug_multiplier):
                aug_img = transform(image=img)['image']
                aug_img = np.array(aug_img, dtype=np.uint8) 
                aug_out_path = str(out_label_dir / f"{unique_base_name}_aug_{i}.jpg")
                
                success, buffer = cv2.imencode('.jpg', aug_img)
                if success: buffer.tofile(aug_out_path)


print("Building dataset...")
data = []
for starch_dir in DATA_DIR.iterdir():
    if not starch_dir.is_dir(): continue
    starch_name = LABEL_MAP.get(starch_dir.name, starch_dir.name)
    for mag_dir in starch_dir.iterdir():
        if not mag_dir.is_dir(): continue
        for session_dir in mag_dir.iterdir():
            if not session_dir.is_dir(): continue
            for img_path in session_dir.glob("*.jpg"):
                data.append({
                    "filepath": str(img_path),
                    "label": starch_name,
                    "group_id": f"{starch_name}_{mag_dir.name}_{session_dir.name}"
                })

df = pd.DataFrame(data)
print(f"Found {len(df)} 640x480 images.")

for fold_idx, current_seed in enumerate(SEEDS):
    print(f"\n--- FOLD {fold_idx + 1}/{len(SEEDS)} (Seed: {current_seed}) ---")
    
    random.seed(current_seed)
    np.random.seed(current_seed)
    
    fold_out_dir = BASE_OUTPUT_DIR / f"fold_{fold_idx}"
    fold_out_dir.mkdir(parents=True, exist_ok=True)
    
    gss_test = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=current_seed)
    train_val_idx, test_idx = next(gss_test.split(df, groups=df['group_id']))
    df_train_val = df.iloc[train_val_idx].reset_index(drop=True)
    df_test = df.iloc[test_idx].copy()

    gss_val = GroupShuffleSplit(n_splits=1, test_size=0.176, random_state=current_seed)
    train_idx, val_idx = next(gss_val.split(df_train_val, groups=df_train_val['group_id']))
    df_train = df_train_val.iloc[train_idx].copy()
    df_val = df_train_val.iloc[val_idx].copy()

    process_and_save(df_train, "train", fold_out_dir, augment=True, aug_multiplier=3)
    process_and_save(df_val, "val", fold_out_dir, augment=False)
    process_and_save(df_test, "test", fold_out_dir, augment=False)

print("\nFinished.")